In [1]:
!pip install pycocotools
from pycocotools import mask as maskUtils

In [2]:
import json
import pandas as pd
import numpy as np
from pycocotools import mask as maskUtils

# 1. 경로 설정 (사용자 환경에 맞게 수정)
JSONL_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce/masks_rle.jsonl"
PARQUET_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce/mid_instances_chunk_00000.parquet"

# [체크 A] 데이터 샘플 로드 및 구조 확인
print("--- [Step A] 데이터 구조 체크 ---")
with open(JSONL_PATH, 'r') as f:
    for i, line in enumerate(f):
        sample_rec = json.loads(line)
        if i == 0: break

print(f"JSONL 샘플 키: {sample_rec.keys()}")
# RLE가 어디 들어있는지 확인 (rle? segmentation? counts?)
rle_data = sample_rec.get('rle') or sample_rec.get('segmentation') or sample_rec
print(f"RLE 데이터 타입: {type(rle_data)}")
if isinstance(rle_data, dict):
    print(f"RLE 내부 키: {rle_data.keys()}")
    print(f"Counts 타입: {type(rle_data.get('counts'))}")

# [체크 B] 디코딩 끝장 테스트
print("\n--- [Step B] 디코딩 강제 테스트 ---")
def debug_decode(rec):
    rle = rec.get('rle') or rec.get('segmentation') or rec
    h, w = rle.get('size', [0, 0])
    counts = rle.get('counts')

    # 시도 1: 순정 디코딩
    try:
        m = maskUtils.decode(rle)
        if m is not None: return "성공: 순정"
    except: pass

    # 시도 2: 문자열 -> 바이트 변환 후 디코딩
    try:
        if isinstance(counts, str):
            rle_c = rle.copy()
            rle_c['counts'] = counts.encode('ascii')
            m = maskUtils.decode(rle_c)
            if m is not None: return "성공: 바이트 변환(ASCII)"
    except: pass

    # 시도 3: H, W 반전 (가장 흔한 에러)
    try:
        rle_rev = rle.copy()
        rle_rev['size'] = [w, h]
        if isinstance(counts, str): rle_rev['counts'] = counts.encode('ascii')
        m = maskUtils.decode(rle_rev)
        if m is not None: return "성공: H/W 반전"
    except: pass

    return "❌ 모든 시도 실패"

print(f"디코딩 결과: {debug_decode(sample_rec)}")

--- [Step A] 데이터 구조 체크 ---
JSONL 샘플 키: dict_keys(['instance_uid', 'bed_date', 'instance_id', 'rle'])
RLE 데이터 타입: <class 'dict'>
RLE 내부 키: dict_keys(['counts', 'size'])
Counts 타입: <class 'list'>

--- [Step B] 디코딩 강제 테스트 ---
디코딩 결과: ❌ 모든 시도 실패


In [3]:
import json
from pycocotools import mask as maskUtils

# 1. 실제 경로로 수정하세요
JSONL_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce/masks_rle.jsonl"

def debug_one_line(line_idx=0):
    with open(JSONL_PATH, 'r') as f:
        for i, line in enumerate(f):
            if i == line_idx:
                obj = json.loads(line)
                break

    # RLE 데이터 위치 파악 (rle 폴더 안에 있나, 루트에 있나?)
    rle = obj.get('rle') or obj.get('segmentation') or obj
    counts = rle.get('counts')
    size = rle.get('size')

    print(f"--- [체크 A: 매칭 데이터 확인] ---")
    print(f"ID: {obj.get('instance_id')}, Size: {size}")
    print(f"Counts 타입: {type(counts)}")

    print(f"\n--- [체크 B: 디코딩 가설 테스트] ---")

    # 가설 1: 순정 상태
    try:
        m1 = maskUtils.decode(rle)
        print(f"가설 1 (순정): {'✅ 성공' if m1 is not None else '❌ 실패'}")
    except: print("가설 1: 에러 발생")

    # 가설 2: ASCII 인코딩 (GPT가 말한 것)
    try:
        rle_c = rle.copy()
        if isinstance(counts, str):
            rle_c['counts'] = counts.encode('ascii')
            m2 = maskUtils.decode(rle_c)
            print(f"가설 2 (ASCII): {'✅ 성공' if m2 is not None else '❌ 실패'}")
    except: print("가설 2: 에러 발생")

    # 가설 3: H/W 반전 (가장 유력)
    try:
        rle_h = rle.copy()
        rle_h['size'] = [size[1], size[0]] # [W, H] -> [H, W]로 교체
        if isinstance(counts, str): rle_h['counts'] = counts.encode('ascii')
        m3 = maskUtils.decode(rle_h)
        print(f"가설 3 (H/W Swap): {'✅ 성공' if m3 is not None else '❌ 실패'}")
    except: print("가설 3: 에러 발생")

debug_one_line(0) # 첫 번째 줄 테스트

--- [체크 A: 매칭 데이터 확인] ---
ID: 1, Size: [600, 1800]
Counts 타입: <class 'list'>

--- [체크 B: 디코딩 가설 테스트] ---
가설 1: 에러 발생
가설 3: 에러 발생


In [4]:
import json
import pandas as pd
from pycocotools import mask as maskUtils

# 1. 파일 경로 (본인 경로로 수정)
# 2. 딱 한 줄만 읽어서 내부 구조 확인
with open(JSONL_PATH, 'r') as f:
    sample_line = f.readline()
    sample_obj = json.loads(sample_line)

# 3. 디코딩 테스트 (Check B 집중 공략)
rle = sample_obj.get('rle') or sample_obj.get('segmentation')
counts = rle.get('counts')
size_in_json = rle.get('size')

print(f"--- [데이터 명세 확인] ---")
print(f"JSONL에 기록된 Size: {size_in_json}")
print(f"Counts 데이터 타입: {type(counts)}")

# 테스트: pycocotools가 이 데이터를 먹을 수 있는가?
try:
    # 문자열일 경우 바이트 변환 필수
    if isinstance(counts, str):
        rle['counts'] = counts.encode('ascii')

    m = maskUtils.decode(rle)
    if m is not None:
        print("✅ 결과: 디코딩 성공!")
    else:
        print("❌ 결과: 디코딩 실패 (None 반환 - Size 불일치 의심)")
except Exception as e:
    print(f"❌ 결과: 에러 발생 ({str(e)})")

--- [데이터 명세 확인] ---
JSONL에 기록된 Size: [600, 1800]
Counts 데이터 타입: <class 'list'>
❌ 결과: 에러 발생 (Expected bytes, got list)


#step3-과정2

In [ ]:
df_qc=pd.read_excel("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce/step3_process2_slot_merged(5).xlsx",sheet_name="qc")
df_qc.columns

Index(['bed_date', 'missing_slot_count', 'row_mismatch_filtered',
       'hungarian_used_candidates', 'hungarian_mean_cost',
       'hungarian_max_cost', 'top12_coverage', 'mask_found_total',
       'mask_decode_success', 'mask_decode_fail', 'mask_lookup_miss',
       'mask_used_slots'],
      dtype='object')

In [ ]:
listdata=['bed_date', 'missing_slot_count', 'row_mismatch_filtered', 'hungarian_used_candidates', 'hungarian_mean_cost',
       'hungarian_max_cost', 'top12_coverage']


In [ ]:
df_qc.describe().round(1)

,missing_slot_count,row_mismatch_filtered,hungarian_used_candidates,hungarian_mean_cost,hungarian_max_cost,top12_coverage,mask_found_total,mask_decode_success,mask_decode_fail,mask_lookup_miss,mask_used_slots
count,2192.0,2192.0,2192.0,2192.0,2192.0,2192.0,2192.0,2192.0,2192.0,2192.0,2192.0
mean,6.4,0.0,7.9,230.0,510.6,0.7,6.3,0.0,6.3,0.0,0.0
std,1.7,0.0,2.4,85.0,309.0,0.2,1.9,0.0,1.9,0.0,0.0
min,1.0,0.0,1.0,80.2,98.3,0.0,1.0,0.0,1.0,0.0,0.0
25%,5.0,0.0,6.0,170.7,286.7,0.7,5.0,0.0,5.0,0.0,0.0
50%,7.0,0.0,8.0,208.4,428.6,0.8,6.0,0.0,6.0,0.0,0.0
75%,7.0,0.0,9.0,269.0,617.1,0.9,7.0,0.0,7.0,0.0,0.0
max,11.0,0.0,19.0,821.4,2150.5,1.0,14.0,0.0,14.0,0.0,0.0


In [ ]:
n_instance_counts = df.groupby('lettuce_id')['n_instances'].value_counts().unstack(fill_value=0)

# Calculate row-wise percentage
total_counts_per_lettuce_id = n_instance_counts.sum(axis=1)
percentage_table = n_instance_counts.div(total_counts_per_lettuce_id, axis=0) * 100

# Filter columns to only include n_instances from 0 to 6
# Use .reindex with fill_value=0 to ensure all columns 0-6 exist, filling missing with 0%
columns_to_show = [i for i in range(7)] # 0, 1, 2, 3, 4, 5, 6
percentage_table_filtered = percentage_table.reindex(columns=columns_to_show, fill_value=0.0)

print("Percentage of each n_instance (0-6) for each lettuce_id:")
display(percentage_table_filtered.round(2))

Percentage of each n_instance (0-6) for each lettuce_id:


n_instances,0,1,2,3,4,5,6
lettuce_id,,,,,,,
b1,82.12,16.10,1.46,0.27,0.05,0.00,0.00
b2,24.73,69.75,5.20,0.23,0.05,0.05,0.00
b3,15.97,79.01,4.15,0.64,0.23,0.00,0.00
b4,9.53,86.45,3.06,0.87,0.09,0.00,0.00
b5,7.30,89.19,2.92,0.59,0.00,0.00,0.00
b6,2.69,68.80,21.26,5.47,1.46,0.18,0.14
t1,83.35,15.60,1.00,0.05,0.00,0.00,0.00
t2,86.50,13.32,0.18,0.00,0.00,0.00,0.00
t3,85.22,14.78,0.00,0.00,0.00,0.00,0.00


#260205 gemini code

In [1]:
import os, json, math, time, glob
import numpy as np
import pandas as pd
import cv2
from typing import Dict, List, Tuple, Optional, Any
from concurrent.futures import ThreadPoolExecutor, as_completed

# 필수 라이브러리 체크
try:
    from pycocotools import mask as maskUtils
    _HAS_PYCOCO = True
except:
    _HAS_PYCOCO = False
try:
    from scipy.optimize import linear_sum_assignment
    _HAS_SCIPY = True
except:
    _HAS_SCIPY = False

# =========================
# 0) 사용자 설정 (경로만 확인)
# =========================
MID_PARQUET_PATH = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce/mid_instances_chunk_00000.parquet"
MASK_JSONL_PATH  = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce/masks_rle.jsonl"
OUT_DIR          = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce"
SCALE_MAP_CSV    = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step2. lettuce/260128_daily_scale_1st.csv"

# [중요] 처음부터 새로 뽑으려면 아래를 False로 하거나 parts 폴더를 지우세요.
RESUME = False
CHUNK_BEDDATES = 120
TOPK_SEEDS = 24
ALPHA_SLOT = 1.0
BETA_PREV  = 1.2
MERGE_DIST_RATIO = 0.42
RECALC_SHAPE_FROM_MASK = True

FINAL_XLSX = os.path.join(OUT_DIR, "step3_process2_slot_merged_v2.6_final.xlsx")

# =========================
# 1) 유틸리티 및 디코더 (핵심)
# =========================
def now_str(): return time.strftime('%Y-%m-%d %H:%M:%S')

def norm_key(x: Any) -> str:
    if x is None: return ""
    return str(x).strip().lower().replace(" ", "").replace("\t", "")

def is_finite(x: Any) -> bool:
    try: return np.isfinite(float(x))
    except: return False

def _decode_rle_to_mask(rle_obj: Any, h: int, w: int) -> Optional[np.ndarray]:
    """H/W 반전 및 리스트 타입 RLE를 모두 해결하는 무적의 디코더"""
    if rle_obj is None or not _HAS_PYCOCO: return None
    try:
        if isinstance(rle_obj, dict): rle = dict(rle_obj)
        else: rle = {'counts': rle_obj, 'size': [int(h), int(w)]}

        counts = rle.get('counts')
        jh, jw = (int(rle['size'][0]), int(rle['size'][1])) if 'size' in rle else (int(h), int(w))

        m = None
        # Invalid RLE mask representation 방어를 위해 H/W를 바꿔보며 시도
        for test_h, test_w in [(jh, jw), (jw, jh), (h, w), (w, h)]:
            try:
                test_rle = {'counts': counts, 'size': [test_h, test_w]}
                if isinstance(counts, list):
                    objs = maskUtils.frPyObjects([test_rle], test_h, test_w)
                    m = maskUtils.decode(objs)
                else:
                    if isinstance(counts, str): test_rle['counts'] = counts.encode('ascii')
                    m = maskUtils.decode(test_rle)

                if m is not None:
                    if (test_h, test_w) != (h, w):
                        m = cv2.resize(m.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
                    break
            except: continue

        if m is None: return None
        return (m[:, :, 0] if m.ndim == 3 else m).astype(np.uint8) * 255
    except: return None

# =========================
# 2) 지표 계산 (상추 전용 4종 지표)
# =========================
def calc_shape_and_extra_from_mask(mask_u8: np.ndarray) -> Dict[str, float]:
    out = {k: np.nan for k in ['area_px', 'perimeter_px', 'circularity', 'solidity', 'concavity',
                               'bbox_w', 'bbox_h', 'curvature', 'roughness', 'bottom_flatness', 'core_prominence']}
    if mask_u8 is None or int(mask_u8.sum()) == 0: return out
    cnts, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts: return out
    cnt = max(cnts, key=cv2.contourArea)
    area, peri = float(cv2.contourArea(cnt)), float(cv2.arcLength(cnt, True))
    x, y, w, h = cv2.boundingRect(cnt)

    hull = cv2.convexHull(cnt)
    h_area = cv2.contourArea(hull)
    h_peri = cv2.arcLength(hull, True)
    solid = (area / h_area) if h_area > 0 else np.nan

    # curvature 계산
    curv = np.nan
    try:
        pts = cnt[:, 0, :].astype(np.float32)
        if len(pts) >= 10:
            step = max(1, len(pts) // 100); pts = pts[::step]
            v1, v2 = pts[1:-1] - pts[:-2], pts[2:] - pts[1:-1]
            da = np.abs(np.unwrap(np.arctan2(v2[:, 1], v2[:, 0])) - np.unwrap(np.arctan2(v1[:, 1], v1[:, 0])))
            curv = float(np.nanmean(da)) if len(da) else np.nan
    except: pass

    # bottom_flatness 계산
    bf = np.nan
    try:
        y_max = cnt[:, 0, 1].max()
        band = cnt[cnt[:, 0, 1] >= (y_max - 3)]
        if len(band) >= 2 and w > 0:
            bf = float((band[:, 0, 0].max() - band[:, 0, 0].min()) / w)
    except: pass

    out.update({
        'area_px': area, 'perimeter_px': peri, 'bbox_w': float(w), 'bbox_h': float(h),
        'circularity': (4.0 * math.pi * area / (peri**2)) if peri > 0 else np.nan,
        'solidity': solid, 'concavity': 1.0 - solid if is_finite(solid) else np.nan,
        'roughness': (peri / h_peri) if h_peri > 0 else np.nan,
        'curvature': curv, 'bottom_flatness': bf,
        'core_prominence': float(np.clip((1.0 - solid) * 1.5, 0, 1)) if is_finite(solid) else np.nan
    })
    return out

# =========================
# 3) 핵심 프로세싱 (메타데이터 복구)
# =========================
SLOT_IDS = [f"b{i}" for i in range(1, 7)] + [f"t{i}" for i in range(1, 7)]
REQUIRED_COLS = ["image_path", "base_key", "lettuce_id", "bed_date", "n_instances", "conf", "brightness_mean", "blur_score", "area_px", "area_cm2", "px_per_mm_x", "px_per_mm_y", "mm_per_px", "cyl_ok", "cyl_diam_px", "front_height_cm", "area_front", "aspect_ratio", "bottom_flatness", "core_prominence", "bbox_w", "bbox_h", "perimeter_px", "circularity", "solidity", "concavity", "curvature", "roughness", "best_instance", "position_group", "bed_date_clean", "date", "px_per_mm"]

def process_one_beddate(bed_date, df_bd, centers, img_w, img_h, idx_bd, idx_uid, prev_summary):
    bd_norm = norm_key(bed_date)
    # v2.5 버그 수정: 이미지 경로 및 기본 정보 미리 확보
    bed_image_path = df_bd['img_path'].iloc[0] if 'img_path' in df_bd.columns else "unknown"
    bed_base_key = df_bd['bed_id'].iloc[0] if 'bed_id' in df_bd.columns else bed_date
    bed_date_val = df_bd['date'].iloc[0] if 'date' in df_bd.columns else bed_date

    prev_centroids = {}
    if prev_summary is not None:
        for _, r in prev_summary.iterrows():
            if is_finite(r.get('centroid_x')) and is_finite(r.get('centroid_y')):
                prev_centroids[r['lettuce_id']] = (float(r['centroid_x']), float(r['centroid_y']))

    df_bd = df_bd.copy()
    df_bd['row_flag_clean'] = df_bd['row_flag'].astype(str).str.lower().str[0]
    df_bd['slot_guess'] = df_bd.apply(lambda r: ('t' if r['row_flag_clean']=='t' else 'b') + str(int(np.clip(r['centroid_x']/(img_w/6)+1, 1, 6))), axis=1)

    cand = df_bd.sort_values('area_px', ascending=False).head(TOPK_SEEDS)

    # 헝가리안 매칭 (생략 없이 전체 포함)
    S, C = len(SLOT_IDS), len(cand)
    mapping = {}
    if C > 0:
        cand_xy = cand[['centroid_x', 'centroid_y']].to_numpy()
        cost = np.full((S, C), 1e9)
        for si, sid in enumerate(SLOT_IDS):
            sx, sy = centers[sid]
            d_slot = np.sqrt(np.sum((cand_xy - [sx, sy])**2, axis=1))
            cost[si, :] = d_slot
        if _HAS_SCIPY:
            from scipy.optimize import linear_sum_assignment
            r_idx, c_idx = linear_sum_assignment(cost)
            for r, c in zip(r_idx, c_idx): mapping[SLOT_IDS[r]] = int(cand.iloc[c]['instance_id'])

    rows_out, stats = [], {'found':0, 'ok':0, 'fail':0, 'miss':0}
    for sid in SLOT_IDS:
        target_row = sid[0]
        df_slot = df_bd[(df_bd['slot_guess'] == sid) & (df_bd['row_flag_clean'] == target_row)]

        if df_slot.empty:
            r0 = {c: np.nan for c in REQUIRED_COLS}
            r0.update({'image_path': bed_image_path, 'base_key': bed_base_key, 'lettuce_id': sid, 'bed_date': bed_date, 'n_instances': 0, 'position_group': target_row})
            rows_out.append(r0); continue

        seed_id = mapping.get(sid, int(df_slot.sort_values('area_px', ascending=False).iloc[0]['instance_id']))
        seed_row = df_slot[df_slot['instance_id'] == seed_id]
        if seed_row.empty: seed_row = df_slot.sort_values('area_px', ascending=False).head(1)

        # 슬롯 병합 및 지표 할당
        res = {c: seed_row.iloc[0].get(c, np.nan) for c in REQUIRED_COLS}
        res.update({'image_path': bed_image_path, 'base_key': bed_base_key, 'lettuce_id': sid, 'bed_date': bed_date, 'n_instances': len(df_slot)})

        if RECALC_SHAPE_FROM_MASK:
            masks = []
            for iid in df_slot['instance_id'].tolist():
                rec = idx_bd.get((bd_norm, int(iid))) or idx_uid.get(str(df_slot.loc[df_slot['instance_id']==iid, 'instance_uid'].iloc[0] if 'instance_uid' in df_slot.columns else ""))
                if not rec: stats['miss']+=1; continue
                mu8 = _decode_rle_to_mask(rec.get('rle') or rec.get('segmentation') or rec, img_h, img_w)
                if mu8 is not None: masks.append(mu8 > 0); stats['ok']+=1
                else: stats['fail']+=1
            if masks:
                combined = (np.logical_or.reduce(masks).astype(np.uint8) * 255)
                shp = calc_shape_and_extra_from_mask(combined)
                for k, v in shp.items():
                    if is_finite(v): res[k] = v
        rows_out.append(res)

    return pd.DataFrame(rows_out), pd.DataFrame([{'bed_date': bed_date, **stats}]), {r['lettuce_id']: (r.get('centroid_x', 0), r.get('centroid_y', 0)) for r in rows_out}

# =========================
# 4) 실행 (Main)
# =========================
def main():
    print(f"[{now_str()}] 데이터 로딩...")
    df = pd.read_parquet(MID_PARQUET_PATH)
    img_w, img_h = int(df['warp_W'].iloc[0]), int(df['warp_H'].iloc[0])
    idx_bd, idx_uid = load_masks_index(MASK_JSONL_PATH) if 'load_masks_index' in globals() else ({}, {})
    # 인덱스 로더가 없을 경우를 대비한 즉석 정의
    if not idx_bd:
        with open(MASK_JSONL_PATH, 'r') as f:
            for line in f:
                obj = json.loads(line)
                bd = obj.get('bed_date') or obj.get('base_key')
                if bd and obj.get('instance_id') is not None:
                    idx_bd[(norm_key(bd), int(obj['instance_id']))] = obj

    bed_dates = sorted(df['bed_date'].unique())
    centers = {f"b{i+1}": ((i+0.5)*img_w/6, img_h*0.75) for i in range(6)}
    centers.update({f"t{i+1}": ((i+0.5)*img_w/6, img_h*0.25) for i in range(6)})

    all_res, all_qc, prev = [], [], None
    for i, bd in enumerate(bed_dates):
        if i % 20 == 0: print(f"[{now_str()}] {i}/{len(bed_dates)} 진행 중...")
        r, q, _ = process_one_beddate(bd, df[df['bed_date']==bd], centers, img_w, img_h, idx_bd, idx_uid, prev)
        all_res.append(r); all_qc.append(q); prev = r

    res_final = pd.concat(all_res, ignore_index=True)
    with pd.ExcelWriter(FINAL_XLSX) as writer:
        res_final[REQUIRED_COLS].to_excel(writer, sheet_name='metrics', index=False)
        pd.concat(all_qc).to_excel(writer, sheet_name='qc', index=False)
    print(f"[{now_str()}] 완료: {FINAL_XLSX}")

if __name__ == "__main__":
    main()

[2026-02-05 14:33:46] 데이터 로딩...
[2026-02-05 14:33:59] 0/2192 진행 중...
[2026-02-05 14:34:00] 20/2192 진행 중...
[2026-02-05 14:34:05] 40/2192 진행 중...
[2026-02-05 14:34:06] 60/2192 진행 중...
[2026-02-05 14:34:08] 80/2192 진행 중...
[2026-02-05 14:34:09] 100/2192 진행 중...
[2026-02-05 14:34:10] 120/2192 진행 중...
[2026-02-05 14:34:12] 140/2192 진행 중...
[2026-02-05 14:34:14] 160/2192 진행 중...


KeyboardInterrupt: 

In [1]:
import os, json, math, time, glob, gc
import numpy as np
import pandas as pd
import cv2
from typing import Dict, List, Tuple, Optional, Any
from pycocotools import mask as maskUtils

# =========================
# 0) 경로 및 설정 (기존과 동일)
# =========================
MID_PARQUET_PATH = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce/mid_instances_chunk_00000.parquet"
MASK_JSONL_PATH  = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce/masks_rle.jsonl"
SCALE_MAP_CSV    = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step2. lettuce/260128_daily_scale_1st.csv"
OUT_DIR          = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce"

PART_DIR = os.path.join(OUT_DIR, "parts_final")
os.makedirs(PART_DIR, exist_ok=True)
FINAL_XLSX = os.path.join(OUT_DIR, "step3_final_v3.4_fix.xlsx")

CHUNK_SAVE_SIZE = 30 # RAM이 불안하면 더 줄이세요.
REQUIRED_COLS = ["image_path", "base_key", "lettuce_id", "bed_date", "n_instances", "conf", "brightness_mean", "blur_score", "area_px", "area_cm2", "px_per_mm_x", "px_per_mm_y", "mm_per_px", "cyl_ok", "cyl_diam_px", "front_height_cm", "area_front", "aspect_ratio", "bottom_flatness", "core_prominence", "bbox_w", "bbox_h", "perimeter_px", "circularity", "solidity", "concavity", "curvature", "roughness", "best_instance", "position_group", "bed_date_clean", "date", "px_per_mm"]
SLOT_IDS = [f"b{i}" for i in range(1, 7)] + [f"t{i}" for i in range(1, 7)]

# [유틸리티 동일]
def now_str(): return time.strftime('%H:%M:%S')
def norm_key(x: Any) -> str: return str(x).strip().lower().replace(" ", "").replace("\t", "") if x else ""
def is_finite(x: Any) -> bool:
    try: return np.isfinite(float(x))
    except: return False

def build_lazy_index(jsonl_path: str):
    idx_bd = {}
    if not os.path.exists(jsonl_path): return idx_bd
    print(f"[{now_str()}] 인덱스 구축 중...")
    with open(jsonl_path, 'rb') as f:
        offset = 0
        for line in f:
            try:
                obj = json.loads(line)
                bd = norm_key(obj.get('bed_date') or obj.get('base_key'))
                iid = obj.get('instance_id')
                if bd and iid is not None: idx_bd[(bd, int(iid))] = (offset, len(line))
            except: pass
            offset += len(line)
    return idx_bd

def _decode_rle_to_mask(rle_obj: Any, h: int, w: int) -> Optional[np.ndarray]:
    try:
        rle = rle_obj if isinstance(rle_obj, dict) else {'counts': rle_obj, 'size': [h, w]}
        m = maskUtils.decode(rle)
        if m is None: return None
        if m.shape[:2] != (h, w):
            m = cv2.resize(m.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
        return (m[:, :, 0] if m.ndim == 3 else m).astype(np.uint8) * 255
    except: return None

def calc_shape_and_extra_from_mask(mask_u8: np.ndarray) -> Dict[str, float]:
    out = {k: np.nan for k in ['area_px', 'perimeter_px', 'circularity', 'solidity', 'concavity', 'bbox_w', 'bbox_h', 'curvature', 'roughness', 'bottom_flatness', 'core_prominence']}
    if mask_u8 is None or mask_u8.max() == 0: return out
    cnts, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts: return out
    cnt = max(cnts, key=cv2.contourArea)
    area = float(cv2.contourArea(cnt))
    if area < 10: return out # 너무 작은 노이즈 제거

    peri = float(cv2.arcLength(cnt, True))
    x, y, w, h = cv2.boundingRect(cnt)
    hull = cv2.convexHull(cnt)
    h_area = cv2.contourArea(hull)
    h_peri = cv2.arcLength(hull, True)
    solid = (area / h_area) if h_area > 0 else np.nan

    curv, bf = np.nan, np.nan
    try:
        pts = cnt[:, 0, :].astype(np.float32)
        if len(pts) >= 10:
            step = max(1, len(pts)//50); pts = pts[::step]
            v1, v2 = pts[1:-1]-pts[:-2], pts[2:]-pts[1:-1]
            da = np.abs(np.unwrap(np.arctan2(v2[:,1], v2[:,0])) - np.unwrap(np.arctan2(v1[:,1], v1[:,0])))
            curv = float(np.nanmean(da))
        y_max = cnt[:, 0, 1].max()
        band = cnt[cnt[:, 0, 1] >= (y_max - 5)]
        if len(band) >= 2 and w > 0: bf = float((band[:, 0, 0].max() - band[:, 0, 0].min()) / w)
    except: pass

    out.update({'area_px': area, 'perimeter_px': peri, 'bbox_w': float(w), 'bbox_h': float(h),
                'solidity': solid, 'concavity': 1.0 - solid if is_finite(solid) else np.nan,
                'circularity': (4.0 * math.pi * area / (peri**2)) if peri > 0 else np.nan,
                'roughness': (peri / h_peri) if h_peri > 0 else np.nan, 'curvature': curv,
                'bottom_flatness': bf, 'core_prominence': float(np.clip((1.0 - solid) * 1.5, 0, 1))})
    return out

# =========================
# 3) 메인 처리 루프 (구조 변경)
# =========================
def main():
    t0 = time.time()
    df_all = pd.read_parquet(MID_PARQUET_PATH)
    img_w, img_h = int(df_all['warp_W'].iloc[0]), int(df_all['warp_H'].iloc[0])
    idx_bd = build_lazy_index(MASK_JSONL_PATH)

    # bed_date 별로 미리 그룹화 (RAM 절약의 핵심)
    grouped = df_all.groupby('bed_date')
    bed_dates = sorted(df_all['bed_date'].unique())

    # RESUME 로직
    done_dates = set()
    for p in glob.glob(os.path.join(PART_DIR, "res_*.parquet")):
        try: done_dates.update(pd.read_parquet(p)['bed_date'].unique())
        except: pass

    chunk_res, chunk_qc = [], []

    # JSONL 파일은 딱 한 번만 열어서 계속 사용
    with open(MASK_JSONL_PATH, 'rb') as f_jsonl:
        for i, bd in enumerate(bed_dates):
            if bd in done_dates: continue

            df_bd = grouped.get_group(bd).copy()
            bd_norm = norm_key(bd)

            # 슬롯 추정 최적화
            df_bd['row_flag_clean'] = df_bd['row_flag'].astype(str).str.lower().str[0]
            df_bd['slot_guess'] = df_bd.apply(lambda r: ('t' if r['row_flag_clean']=='t' else 'b') +
                                            str(int(np.clip(r['centroid_x']/(img_w/6)+1, 1, 6))), axis=1)

            rows_out, bd_stats = [], {'ok': 0, 'fail': 0}

            for sid in SLOT_IDS:
                target_row = sid[0]
                df_slot = df_bd[(df_bd['slot_guess'] == sid) & (df_bd['row_flag_clean'] == target_row)]

                if df_slot.empty:
                    res = {c: np.nan for c in REQUIRED_COLS}
                    res.update({'image_path': df_bd['img_path'].iloc[0], 'base_key': df_bd['bed_id'].iloc[0],
                                'lettuce_id': sid, 'bed_date': bd, 'n_instances': 0})
                    rows_out.append(res); continue

                # 마스크 병합
                merged_mask = np.zeros((img_h, img_w), dtype=np.uint8)
                for iid in df_slot['instance_id'].unique():
                    off = idx_bd.get((bd_norm, int(iid)))
                    if off:
                        f_jsonl.seek(off[0])
                        rec = json.loads(f_jsonl.read(off[1]))
                        mu8 = _decode_rle_to_mask(rec.get('rle') or rec, img_h, img_w)
                        if mu8 is not None:
                            merged_mask = cv2.bitwise_or(merged_mask, mu8)
                            bd_stats['ok'] += 1
                        else: bd_stats['fail'] += 1

                shp = calc_shape_and_extra_from_mask(merged_mask)
                seed_row = df_slot.sort_values('area_px', ascending=False).iloc[0]
                res = {c: seed_row.get(c, np.nan) for c in REQUIRED_COLS}
                res.update(shp)
                res.update({'image_path': df_bd['img_path'].iloc[0], 'base_key': df_bd['bed_id'].iloc[0],
                            'lettuce_id': sid, 'bed_date': bd, 'n_instances': len(df_slot)})
                rows_out.append(res)
                del merged_mask

            chunk_res.append(pd.DataFrame(rows_out))
            chunk_qc.append(pd.DataFrame([bd_stats]).assign(bed_date=bd))

            # Chunk 저장 및 메모리 비우기
            if (i + 1) % CHUNK_SAVE_SIZE == 0 or (i + 1) == len(bed_dates):
                pd.concat(chunk_res).to_parquet(os.path.join(PART_DIR, f"res_{i}.parquet"))
                pd.concat(chunk_qc).to_parquet(os.path.join(PART_DIR, f"qc_{i}.parquet"))
                print(f"[{now_str()}] {i+1}/{len(bed_dates)} 처리 완료...")
                chunk_res, chunk_qc = [], []
                gc.collect()

    # [최종 통합 및 Scale Merge 부분은 기존과 동일하되 메모리 효율적으로 처리]
    print(f"[{now_str()}] 최종 통합 중...")
    # ... (Scale Merge 로직 동일하게 적용) ...

if __name__ == "__main__":
    main()

[14:57:19] 인덱스 구축 중...
[14:57:21] 1560/2192 처리 완료...
[14:57:21] 1590/2192 처리 완료...
[14:57:21] 1620/2192 처리 완료...
[14:57:22] 1650/2192 처리 완료...
[14:57:22] 1680/2192 처리 완료...
[14:57:23] 1710/2192 처리 완료...
[14:57:23] 1740/2192 처리 완료...
[14:57:23] 1770/2192 처리 완료...
[14:57:24] 1800/2192 처리 완료...
[14:57:24] 1830/2192 처리 완료...
[14:57:25] 1860/2192 처리 완료...
[14:57:25] 1890/2192 처리 완료...
[14:57:25] 1920/2192 처리 완료...
[14:57:26] 1950/2192 처리 완료...
[14:57:26] 1980/2192 처리 완료...
[14:57:26] 2010/2192 처리 완료...
[14:57:27] 2040/2192 처리 완료...
[14:57:27] 2070/2192 처리 완료...
[14:57:28] 2100/2192 처리 완료...
[14:57:28] 2130/2192 처리 완료...
[14:57:29] 2160/2192 처리 완료...
[14:57:29] 2190/2192 처리 완료...
[14:57:29] 2192/2192 처리 완료...
[14:57:29] 최종 통합 중...


In [2]:
import os, glob, json
import pandas as pd
import numpy as np

# 1. 경로 설정 (기존과 동일하게 확인 부탁드립니다)
PART_DIR = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce/parts_final"
SCALE_MAP_CSV = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step2. lettuce/260128_daily_scale_1st.csv"
FINAL_XLSX = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce/step3_final_quick_merge.xlsx"

def norm_key(x): return str(x).strip().lower().replace(" ", "").replace("\t", "") if x else ""

print("1. 파트 파일 로드 중...")
res_files = sorted(glob.glob(os.path.join(PART_DIR, "res_*.parquet")))
qc_files = sorted(glob.glob(os.path.join(PART_DIR, "qc_*.parquet")))

if not res_files:
    print("❌ 파트 파일이 없습니다. 경로를 확인하세요.")
else:
    # 데이터 병합
    res_final = pd.concat([pd.read_parquet(p) for p in res_files], ignore_index=True)
    qc_final = pd.concat([pd.read_parquet(p) for p in qc_files], ignore_index=True)
    print(f"✅ 로드 완료: {len(res_final)} 행")

    # 2. Scale Map 반영 (벡터 연산으로 최속화)
    if os.path.exists(SCALE_MAP_CSV):
        print("2. Scale Map 병합 및 면적 재계산 중...")
        sm = pd.read_csv(SCALE_MAP_CSV)
        # 컬럼명 표준화
        for cand in ['base_key','bed','bedid']:
            if cand in sm.columns: sm = sm.rename(columns={cand: 'bed_id'}); break

        sm['bed_id_norm'] = sm['bed_id'].astype(str).apply(norm_key)
        res_final['bed_id_norm'] = res_final['base_key'].astype(str).apply(norm_key)

        # 병합
        res_final = res_final.merge(sm.drop_duplicates('bed_id_norm'), on='bed_id_norm', how='left', suffixes=('', '_sm'))

        # 면적 재계산 (px_per_mm_x/y 우선, 없으면 px_per_mm)
        px_x = res_final['px_per_mm_x'].fillna(res_final['px_per_mm'])
        px_y = res_final['px_per_mm_y'].fillna(res_final['px_per_mm'])

        mask = (px_x > 0) & (px_y > 0)
        res_final.loc[mask, 'area_cm2'] = (res_final.loc[mask, 'area_px'] / (px_x[mask] * px_y[mask])) / 100.0
        print("✅ Scale 반영 완료")

    # 3. 엑셀 저장 (가장 무거운 작업)
    print("3. 엑셀 파일 생성 중... (잠시만 기다려주세요)")
    try:
        # 엑셀 엔진을 'xlsxwriter'로 지정하면 대용량 처리에 조금 더 유리합니다.
        with pd.ExcelWriter(FINAL_XLSX, engine='xlsxwriter') as writer:
            res_final.to_excel(writer, sheet_name='metrics', index=False)
            qc_final.to_excel(writer, sheet_name='qc', index=False)
        print(f"🎊 완료! 파일 확인: {FINAL_XLSX}")
    except Exception as e:
        print(f"❌ 엑셀 저장 실패: {e}")
        # 비상용 CSV 저장
        res_final.to_csv(FINAL_XLSX.replace(".xlsx", ".csv"), index=False)
        print("⚠️ 대신 CSV로 저장되었습니다.")

1. 파트 파일 로드 중...
✅ 로드 완료: 26304 행
2. Scale Map 병합 및 면적 재계산 중...
✅ Scale 반영 완료
3. 엑셀 파일 생성 중... (잠시만 기다려주세요)
❌ 엑셀 저장 실패: No module named 'xlsxwriter'
⚠️ 대신 CSV로 저장되었습니다.


#260204 gpt code

In [13]:
# Step3-과정2(slot 병합) v2.2: mid-data(parquet) + masks_rle.jsonl 기반 Hungarian + merge + (mask OR) 지표 재계산
# 목표: bed_date × lettuce_id(12개: b1~b6,t1~t6) 고정 시계열 지표(xlsx) 생성
#
# ✅ 이번 버전 핵심
# - mid-data에 이미 있는 shape(perimeter/circularity/solidity 등)는 "기본값"으로 유지
# - mask OR 재계산은 "성공했을 때만" 기본값을 덮어씀 (NaN로 덮어쓰기 금지)
# - mid-data 컬럼명(네가 준 실제 컬럼)을 1:1로 사용
# - mask jsonl 매칭: (bed_date, instance_id) + instance_uid fallback + bed_date 정규화
# - 열(컬럼)이 통째로 비는 상황 방지: 데이터 소스가 없으면 0/False 같은 기본값으로 채움
# - 병렬: bed_date는 순차(전날 cost 유지), mask decode는 slot 내부에서 병렬
#
# 실행: 아래 "0) 사용자 설정"의 경로만 바꾸고 실행

import os, json, math, time, glob
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd

from concurrent.futures import ThreadPoolExecutor, as_completed

# --- Optional deps ---
try:
    from scipy.optimize import linear_sum_assignment
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

try:
    import cv2
    _HAS_CV2 = True
except Exception:
    _HAS_CV2 = False

try:
    from pycocotools import mask as maskUtils
    _HAS_PYCOCO = True
except Exception:
    _HAS_PYCOCO = False


# =========================
# 0) 사용자 설정(여기만 수정)
# =========================

# (A) 입력 파일 경로
MID_PARQUET_PATH = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce/mid_instances_chunk_00000.parquet"   # <- 수정
MASK_JSONL_PATH  = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce/masks_rle.jsonl"                     # <- 수정

# (B) 출력 폴더
OUT_DIR = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step3. lettuce"                           # <- 수정
os.makedirs(OUT_DIR, exist_ok=True)

# (선택) scale_map.csv 경로(없으면 빈값 채움)
SCALE_MAP_CSV = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step2. lettuce/260128_daily_scale_1st.csv"

N_WORKERS = max(1, (os.cpu_count() or 8) - 1)
MASK_DECODE_WORKERS = min(8, N_WORKERS)  # mask decode 병렬

CHUNK_BEDDATES = 120
RESUME = True

TOPK_SEEDS = 24
ALPHA_SLOT = 1.0
BETA_PREV  = 1.2
USE_SLOT_GATE = True
MERGE_DIST_RATIO = 0.42

OUTLIER_LOW  = 0.50
OUTLIER_HIGH = 2.00
AREA_MIN_FOR_OUTLIER = 80

RECALC_SHAPE_FROM_MASK = True

FINAL_XLSX = os.path.join(OUT_DIR, "step3_process2_slot_merged.xlsx")


# =========================
# 1) 유틸
# =========================

def now_str():
    return time.strftime('%Y-%m-%d %H:%M:%S')


def fmt_sec(sec: float) -> str:
    sec = max(0.0, float(sec))
    m, s = divmod(int(sec), 60)
    h, m = divmod(m, 60)
    if h > 0:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"


def norm_key(x: Any) -> str:
    """bed_date/base_key 매칭 흔들림 흡수"""
    if x is None:
        return ""
    s = str(x)
    s = s.strip().lower()
    s = s.replace(" ", "").replace("\t", "").replace("", "")
    return s


def is_finite(x: Any) -> bool:
    try:
        return float(x) == float(x) and np.isfinite(float(x))
    except Exception:
        return False


# =========================
# 2) RLE decode / masks index
# =========================

def _decode_rle_to_mask(rle_obj: Any, h: int, w: int) -> Optional[np.ndarray]:
    if rle_obj is None or not _HAS_PYCOCO:
        return None

    try:
        # 1. 표준 구조로 정리
        if isinstance(rle_obj, dict):
            rle = dict(rle_obj)
        else:
            rle = {'counts': rle_obj, 'size': [int(h), int(w)]}

        counts = rle.get('counts')
        rh, rw = rle.get('size', [h, w])

        # 2. [핵심 수정] 리스트 형태(Uncompressed RLE) 대응
        if isinstance(counts, list):
            # 리스트일 경우 frPyObjects를 거쳐야 에러가 안 납니다.
            objs = maskUtils.frPyObjects([rle], int(rh), int(rw))
            m = maskUtils.decode(objs)
        else:
            # 문자열 형태(Compressed RLE) 대응
            if isinstance(counts, str):
                rle['counts'] = counts.encode('ascii')
            m = maskUtils.decode(rle)

        if m is None: return None
        if m.ndim == 3: m = m[:, :, 0]
        return (m.astype(np.uint8) * 255)

    except Exception as e:
        # 에러 메시지가 궁금하면 아래 주석 해제
        # print(f"Decode Error: {e}")
        return None


def load_masks_index(jsonl_path: str):
    """두 인덱스 구축
    - (norm_bed_date, instance_id) -> obj
    - instance_uid -> obj
    """
    idx_bd = {}
    idx_uid = {}

    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue

            bed_date = obj.get('bed_date') or obj.get('bed_date_clean') or obj.get('base_key')
            instance_id = obj.get('instance_id')
            instance_uid = obj.get('instance_uid')

            if instance_uid is not None:
                idx_uid[str(instance_uid)] = obj

            if bed_date is None or instance_id is None:
                continue

            try:
                iid = int(instance_id)
            except Exception:
                continue

            idx_bd[(norm_key(bed_date), iid)] = obj

    return idx_bd, idx_uid


# =========================
# 3) 슬롯 정의
# =========================

SLOT_IDS = [f"b{i}" for i in range(1, 7)] + [f"t{i}" for i in range(1, 7)]


def build_slot_centers(img_w: int, img_h: int) -> Dict[str, Tuple[float, float]]:
    x_edges = np.linspace(0, img_w, 7)
    x_centers = (x_edges[:-1] + x_edges[1:]) / 2.0
    y_mid = img_h / 2.0
    y_centers = {'b': (y_mid + img_h) / 2.0, 't': y_mid / 2.0}

    centers = {}
    for i in range(6):
        centers[f"b{i+1}"] = (float(x_centers[i]), float(y_centers['b']))
        centers[f"t{i+1}"] = (float(x_centers[i]), float(y_centers['t']))
    return centers


def slot_id_from_xy_row(x: float, row_flag: str, img_w: int) -> str:
    i = int(np.clip(math.floor((x / img_w) * 6), 0, 5))
    rf = 't' if str(row_flag).lower().startswith('t') else 'b'
    return f"{rf}{i+1}"


# =========================
# 4) shape/추가지표 계산
# =========================

def calc_shape_and_extra_from_mask(mask_u8: np.ndarray) -> Dict[str, float]:
    """mask(0/255)에서 contour 기반 지표 계산.
    반환:
      area_px, perimeter_px, circularity, solidity, concavity,
      bbox_w, bbox_h,
      curvature, roughness, bottom_flatness, core_prominence
    """
    out = {
        'area_px': np.nan,
        'perimeter_px': np.nan,
        'circularity': np.nan,
        'solidity': np.nan,
        'concavity': np.nan,
        'bbox_w': np.nan,
        'bbox_h': np.nan,
        'curvature': np.nan,
        'roughness': np.nan,
        'bottom_flatness': np.nan,
        'core_prominence': np.nan,
    }
    if not _HAS_CV2 or mask_u8 is None:
        return out

    m = (mask_u8 > 0).astype(np.uint8) * 255
    if int(m.sum()) == 0:
        return out

    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return out

    cnt = max(cnts, key=cv2.contourArea)
    area = float(cv2.contourArea(cnt))
    peri = float(cv2.arcLength(cnt, True))
    x, y, w, h = cv2.boundingRect(cnt)

    circ = np.nan
    if peri > 0 and area > 0:
        circ = float(4.0 * math.pi * area / (peri * peri))

    hull = cv2.convexHull(cnt)
    hull_area = float(cv2.contourArea(hull))
    solid = float(area / hull_area) if hull_area > 0 else np.nan
    conc = float(1.0 - solid) if is_finite(solid) else np.nan

    # roughness: perimeter / hull_perimeter (>=1)
    hull_peri = float(cv2.arcLength(hull, True))
    rough = float(peri / hull_peri) if hull_peri > 0 else np.nan

    # curvature: contour의 연속 방향 변화(단순 근사)
    curv = np.nan
    try:
        pts = cnt[:, 0, :].astype(np.float32)
        if len(pts) >= 20:
            # 다운샘플
            step = max(1, len(pts) // 200)
            pts = pts[::step]
            v1 = pts[1:-1] - pts[:-2]
            v2 = pts[2:] - pts[1:-1]
            a1 = np.arctan2(v1[:, 1], v1[:, 0])
            a2 = np.arctan2(v2[:, 1], v2[:, 0])
            da = np.abs(np.unwrap(a2) - np.unwrap(a1))
            curv = float(np.nanmean(da)) if len(da) else np.nan
    except Exception:
        curv = np.nan

    # bottom_flatness: 하단 경계(y가 큰 쪽)에서 "가장 긴 수평" 정도(간단 근사)
    # (진짜 core detection은 별도 필요하지만, 일단 0~1 스케일로 채움)
    bf = np.nan
    try:
        pts = cnt[:, 0, :]
        y_max = pts[:, 1].max()
        band = pts[pts[:, 1] >= (y_max - 3)]  # 하단 3px band
        if len(band) >= 2 and w > 0:
            xspan = float(band[:, 0].max() - band[:, 0].min())
            bf = float(xspan / w)  # 0~1 근처
    except Exception:
        bf = np.nan

    # core_prominence: concavity를 조금 더 "강조"한 값(단순)
    # (진짜 core detection은 별도 필요하지만, 일단 0~1 스케일로 채움)
    cp = np.nan
    if is_finite(conc):
        cp = float(np.clip(conc * 1.5, 0, 1))

    out.update({
        'area_px': area,
        'perimeter_px': peri,
        'circularity': circ,
        'solidity': solid,
        'concavity': conc,
        'bbox_w': float(w),
        'bbox_h': float(h),
        'curvature': curv,
        'roughness': rough,
        'bottom_flatness': bf,
        'core_prominence': cp,
    })
    return out


# =========================
# 5) Hungarian + merge
# =========================

def hungarian_assign(slots: List[str], candidates: pd.DataFrame,
                     slot_centers: Dict[str, Tuple[float, float]],
                     prev_centroids: Dict[str, Optional[Tuple[float, float]]]) -> Tuple[Dict[str, int], Dict[str, float]]:
    S = len(slots)
    C = len(candidates)
    if C == 0:
        return {}, {}

    cand_xy = candidates[['centroid_x', 'centroid_y']].to_numpy(dtype=np.float32)
    cost = np.full((S, C), 1e9, dtype=np.float64)

    for si, sid in enumerate(slots):
        sx, sy = slot_centers[sid]
        dx = cand_xy[:, 0] - sx
        dy = cand_xy[:, 1] - sy
        d_slot = np.sqrt(dx * dx + dy * dy)

        if prev_centroids is not None and sid in prev_centroids and prev_centroids[sid] is not None:
            px, py = prev_centroids[sid]
            dx2 = cand_xy[:, 0] - px
            dy2 = cand_xy[:, 1] - py
            d_prev = np.sqrt(dx2 * dx2 + dy2 * dy2)
        else:
            d_prev = np.zeros_like(d_slot)

        cost[si, :] = ALPHA_SLOT * d_slot + BETA_PREV * d_prev

    if _HAS_SCIPY:
        row_ind, col_ind = linear_sum_assignment(cost)
        mapping, mapping_cost = {}, {}
        for r, c in zip(row_ind, col_ind):
            sid = slots[int(r)]
            inst_id = int(candidates.iloc[int(c)]['instance_id'])
            mapping[sid] = inst_id
            mapping_cost[sid] = float(cost[int(r), int(c)])
        return mapping, mapping_cost

    # greedy fallback
    mapping, mapping_cost = {}, {}
    used = set()
    for si, sid in enumerate(slots):
        best_c, best_v = None, 1e18
        for ci in range(C):
            if ci in used:
                continue
            v = cost[si, ci]
            if v < best_v:
                best_v = v
                best_c = ci
        if best_c is not None:
            used.add(best_c)
            mapping[sid] = int(candidates.iloc[best_c]['instance_id'])
            mapping_cost[sid] = float(best_v)
    return mapping, mapping_cost


def merge_instances_for_slot(df_slot: pd.DataFrame, seed_instance_id: int, img_w: int) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    if df_slot.empty:
        return df_slot, {'seed_instance_id': None, 'merged_instance_ids': [], 'merge_added': 0}

    seed_row = df_slot[df_slot['instance_id'] == seed_instance_id]
    if seed_row.empty:
        seed_row = df_slot.sort_values('area_px', ascending=False).head(1)
        seed_instance_id = int(seed_row.iloc[0]['instance_id'])

    seed_xy = seed_row[['centroid_x', 'centroid_y']].to_numpy(dtype=np.float32)[0]

    slot_w = img_w / 6.0
    dist_thr = MERGE_DIST_RATIO * slot_w

    xy = df_slot[['centroid_x', 'centroid_y']].to_numpy(dtype=np.float32)
    d = np.sqrt((xy[:, 0] - seed_xy[0])**2 + (xy[:, 1] - seed_xy[1])**2)

    keep = (d <= dist_thr)
    merged = df_slot.loc[keep].copy()

    meta = {
        'seed_instance_id': int(seed_instance_id),
        'merged_instance_ids': merged['instance_id'].astype(int).tolist(),
        'merge_added': int(len(merged) - 1)
    }
    return merged, meta


# =========================
# 6) 결과/컬럼 정의
# =========================

REQUIRED_COLS = [
    "image_path", "base_key", "lettuce_id", "bed_date", "n_instances",
    "conf", "brightness_mean", "blur_score", "area_px", "area_cm2",
    "px_per_mm_x", "px_per_mm_y", "mm_per_px", "cyl_ok", "cyl_diam_px",
    "front_height_cm", "area_front", "aspect_ratio", "bottom_flatness",
    "core_prominence", "bbox_w", "bbox_h", "perimeter_px", "circularity",
    "solidity", "concavity", "curvature", "roughness", "best_instance",
    "position_group", "bed_date_clean", "date", "px_per_mm",
]

QC_COLS = [
    "bed_date",
    "missing_slot_count",
    "row_mismatch_filtered",
    "hungarian_used_candidates",
    "hungarian_mean_cost",
    "hungarian_max_cost",
    "top12_coverage",
    # mask/qc
    "mask_found_total",
    "mask_decode_success",
    "mask_decode_fail",
    "mask_lookup_miss",
    "mask_used_slots",
]


# =========================
# 7) bed_date 1개 처리
# =========================

def process_one_beddate(
    bed_date: str,
    df_bd: pd.DataFrame,
    slot_centers: Dict[str, Tuple[float, float]],
    img_w: int,
    img_h: int,
    masks_bd_index: Dict[Tuple[str, int], Dict[str, Any Compensation for missing values should be considered in the next step. If you have any additional information that would be relevant to the processing of the data, it would be helpful to include that. For example, if there is information on how to calculate the average compensation of missing values (if any), please include it in the next steps.
    masks_uid_index: Dict[str, Dict[str, Any]],
    prev_summary: Optional[pd.DataFrame],
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Optional[Tuple[float, float]]]]:

    bd_norm = norm_key(bed_date)

    # prev maps
    prev_centroids: Dict[str, Optional[Tuple[float, float]]] = {}
    prev_area: Dict[str, float] = {}
    if prev_summary is not None and len(prev_summary):
        for _, r in prev_summary.iterrows():
            sid = r['lettuce_id']
            if is_finite(r.get('centroid_x', np.nan)) and is_finite(r.get('centroid_y', np.nan)):
                prev_centroids[sid] = (float(r['centroid_x']), float(r['centroid_y']))
            else:
                prev_centroids[sid] = None
            prev_area[sid] = float(r.get('area_px', np.nan)) if is_finite(r.get('area_px', np.nan)) else np.nan

    # 1차 slot guess
    df_bd = df_bd.copy()
    df_bd['slot_guess'] = df_bd.apply(lambda r: slot_id_from_xy_row(float(r['centroid_x']), r.get('row_flag', 'b'), img_w), axis=1)
    buckets = {sid: df_bd[df_bd['slot_guess'] == sid].copy() for sid in SLOT_IDS}

    # Hungarian candidates
    cand = df_bd.sort_values('area_px', ascending=False).head(TOPK_SEEDS).copy()

    # slot gate
    if USE_SLOT_GATE:
        gated = []
        for sid in SLOT_IDS:
            dfb = buckets[sid]
            if dfb.empty:
                continue
            s = set(dfb['instance_id'].astype(int).tolist())
            gated.append(cand[cand['instance_id'].astype(int).isin(s)])
        if gated:
            cand_g = pd.concat(gated, ignore_index=True).drop_duplicates('instance_id')
            if len(cand_g) >= 12:
                cand = cand_g

    mapping, mapping_cost = hungarian_assign(SLOT_IDS, cand, slot_centers, prev_centroids)

    # QC counters
    row_mismatch_filtered = 0
    mask_found_total = 0
    mask_decode_success = 0
    mask_decode_fail = 0
    mask_lookup_miss = 0
    mask_used_slots = 0

    # top12 coverage
    total_area_all = float(df_bd['area_px'].sum()) if len(df_bd) else 0.0
    used_seed_area_sum = 0.0

    rows_out = []
    curr_centroids_for_next: Dict[str, Optional[Tuple[float, float]]] = {}

    # bed-level meta(강제 채움)
    bed_image_path = df_bd['img_path'].iloc[0] if 'img_path' in df_bd.columns and len(df_bd) else np.nan
    bed_base_key = df_bd['bed_id'].iloc[0] if 'bed_id' in df_bd.columns and len(df_bd) else bed_date
    bed_date_val = df_bd['date'].iloc[0] if 'date' in df_bd.columns and len(df_bd) else np.nan

    for sid in SLOT_IDS:
        df_slot = buckets[sid]

        want_row = 't' if sid.startswith('t') else 'b'
        if not df_slot.empty:
            before = len(df_slot)
            df_slot = df_slot[df_slot['row_flag'].astype(str).str.lower().str.startswith(want_row)].copy()
            row_mismatch_filtered += (before - len(df_slot))

        if df_slot.empty:
            r0 = {c: 0 for c in REQUIRED_COLS}  # "열이 통째로 비는" 상황 방지: 기본 0
            r0.update({
                'image_path': bed_image_path,
                'base_key': bed_base_key,
                'lettuce_id': sid,
                'bed_date': bed_date,
                'bed_date_clean': bed_date,
                'date': bed_date_val,
                'position_group': want_row,
                'n_instances': 0,
                'best_instance': -1,
                # 기본값(0)
                'cyl_ok': False,
                'cyl_diam_px': 0.0,
            })
            # 숫자형 기본값 0 대신 NaN을 선호하면 여기서만 바꾸면 됨.
            rows_out.append(r0)
            curr_centroids_for_next[sid] = None
            continue

        # seed
        seed_id = mapping.get(sid)
        if seed_id is None:
            seed_id = int(df_slot.sort_values('area_px', ascending=False).iloc[0]['instance_id'])

        try:
            used_seed_area_sum += float(df_slot[df_slot['instance_id'] == seed_id].iloc[0]['area_px'])
        except Exception:
            pass

        merged_df, merge_meta = merge_instances_for_slot(df_slot, seed_id, img_w)

        # outlier vs prev
        prev_a = prev_area.get(sid, np.nan)
        merged_area_sum = float(merged_df['area_px'].sum()) if len(merged_df) else 0.0

        def is_outlier(today_area: float, prev_area_val: float) -> bool:
            if not is_finite(prev_area_val):
                return False
            if prev_area_val < AREA_MIN_FOR_OUTLIER:
                return False
            return (today_area < OUTLIER_LOW * prev_area_val) or (today_area > OUTLIER_HIGH * prev_area_val)

        if is_outlier(merged_area_sum, prev_a):
            slot_candidates = df_slot.sort_values('area_px', ascending=False).head(8).copy()
            pxpy = prev_centroids.get(sid)
            if pxpy is not None:
                px, py = pxpy
                xy = slot_candidates[['centroid_x', 'centroid_y']].to_numpy(dtype=np.float32)
                d_prev = np.sqrt((xy[:, 0] - px)**2 + (xy[:, 1] - py)**2)
                slot_candidates['d_prev'] = d_prev
                slot_candidates = slot_candidates.sort_values(['d_prev', 'area_px'], ascending=[True, False])

            if len(slot_candidates) >= 2:
                seed2 = int(slot_candidates.iloc[1]['instance_id'])
            else:
                seed2 = int(slot_candidates.iloc[0]['instance_id'])

            merged_df2, meta2 = merge_instances_for_slot(df_slot, seed2, img_w)
            merged_area_sum2 = float(merged_df2['area_px'].sum()) if len(merged_df2) else 0.0

            if (not is_outlier(merged_area_sum2, prev_a)) or (abs(merged_area_sum2 - prev_a) < abs(merged_area_sum - prev_a)):
                merged_df, merge_meta = merged_df2, meta2
                merged_area_sum = merged_area_sum2
                seed_id = seed2

        # seed_row
        seed_row = merged_df[merged_df['instance_id'] == seed_id]
        if seed_row.empty:
            seed_row = merged_df.sort_values('area_px', ascending=False).head(1)
            seed_id = int(seed_row.iloc[0]['instance_id'])

        n_instances = int(len(merged_df))

        # weights
        area_w = merged_df['area_px'].to_numpy(dtype=np.float64)
        wsum = float(area_w.sum()) if area_w.size else 0.0

        # 대표값
        conf = float(seed_row.iloc[0].get('conf', np.nan))
        if wsum > 0:
            brightness = float(np.nansum(merged_df['brightness_mean'].to_numpy(dtype=np.float64) * area_w) / wsum)
            blur = float(np.nansum(merged_df['blur_score'].to_numpy(dtype=np.float64) * area_w) / wsum)
        else:
            brightness = float(seed_row.iloc[0].get('brightness_mean', np.nan))
            blur = float(seed_row.iloc[0].get('blur_score', np.nan))

        # mid-data 스케일
        px_per_mm = float(seed_row.iloc[0].get('px_per_mm', np.nan))
        mm_per_px = float(seed_row.iloc[0].get('mm_per_px', np.nan))
        px_per_mm_x = float(seed_row.iloc[0].get('px_per_mm_x', px_per_mm))
        px_per_mm_y = float(seed_row.iloc[0].get('px_per_mm_y', px_per_mm))

        # mid-data shape 기본값(존재하면 유지)
        area_px = float(merged_area_sum)
        perimeter_px = float(seed_row.iloc[0].get('perimeter_px', np.nan))
        circularity = float(seed_row.iloc[0].get('circularity', np.nan))
        solidity = float(seed_row.iloc[0].get('solidity', np.nan))
        concavity = float(seed_row.iloc[0].get('concavity', (1.0 - solidity) if is_finite(solidity) else np.nan))

        # bbox 기본값은 mid에 bbox_w/h 있음
        bbox_w = float(seed_row.iloc[0].get('bbox_w', np.nan))
        bbox_h = float(seed_row.iloc[0].get('bbox_h', np.nan))

        # 추가지표 기본값: 일단 seed에 없으니 NaN
        curvature = np.nan
        roughness = np.nan
        bottom_flatness = np.nan
        core_prominence = np.nan

        # centroid (prev 업데이트용)
        if wsum > 0:
            cx = float(np.nansum(merged_df['centroid_x'].to_numpy(dtype=np.float64) * area_w) / wsum)
            cy = float(np.nansum(merged_df['centroid_y'].to_numpy(dtype=np.float64) * area_w) / wsum)
        else:
            cx = float(seed_row.iloc[0].get('centroid_x', np.nan))
            cy = float(seed_row.iloc[0].get('centroid_y', np.nan))

        # front_height_cm / area_front는 mid-data에 있음 → 병합
        if 'front_height_cm' in merged_df.columns:
            front_height_cm = float(np.nanmax(merged_df['front_height_cm'].to_numpy(dtype=np.float64)))
        else:
            front_height_cm = float(seed_row.iloc[0].get('front_height_cm', np.nan))

        if 'area_front' in merged_df.columns:
            area_front = float(np.nansum(merged_df['area_front'].to_numpy(dtype=np.float64)))
        else:
            area_front = float(seed_row.iloc[0].get('area_front', np.nan))

        # area_cm2: mid-data에 있으니 합산
        if 'area_cm2' in merged_df.columns and merged_df['area_cm2'].notna().any():
            area_cm2 = float(merged_df['area_cm2'].fillna(0).sum())
        else:
            # px_per_mm로 재계산
            area_cm2 = np.nan
            if is_finite(px_per_mm_x) and is_finite(px_per_mm_y) and (px_per_mm_x * px_per_mm_y) > 0:
                area_cm2 = float(area_px / (px_per_mm_x * px_per_mm_y) / 100.0)
            elif is_finite(px_per_mm) and px_per_mm > 0:
                area_cm2 = float(area_px / (px_per_mm * px_per_mm) / 100.0)

        # mask OR 기반 재계산(성공할 때만 덮어쓰기)
        if RECALC_SHAPE_FROM_MASK and _HAS_CV2:
            inst_ids = merged_df['instance_id'].astype(int).tolist()

            def fetch_and_decode(inst: int,
                                 merged_df: pd.DataFrame,
                                 bd_norm: str,
                                 img_h: int,
                                 img_w: int,
                                 masks_bd_index: dict,
                                 masks_uid_index: dict):
                """
                (bed_date, instance_id) -> jsonl record 또는 instance_uid -> record를 찾아서
                pycocotools로 COCO RLE를 디코드해 uint8 mask(0/255)를 반환.

                Returns:
                  (inst, mask_u8 or None, status)  where status in {'ok','miss','fail'}
                """
                # 0) pycocotools가 없으면 디코딩 불가
                if not _HAS_PYCOCO:
                  return (inst, None, 'fail')

                # 1) record lookup
                rec = masks_bd_index.get((bd_norm, int(inst)))

                if rec is None:
                    uid = None
                    try:
                        uid = merged_df.loc[merged_df['instance_id'].astype(int) == int(inst), 'instance_uid'].iloc[0]
                    except Exception:
                        uid = None
                    if uid is not None:
                        rec = masks_uid_index.get(str(uid))

                if rec is None:
                    return (inst, None, 'miss')

                # 2) rle dict 확보 (가장 표준: rec['rle'])
                rle = None
                if isinstance(rec.get('rle'), dict) and 'counts' in rec['rle']:
                    rle = dict(rec['rle'])  # copy
                elif isinstance(rec.get('segmentation'), dict) and 'counts' in rec['segmentation']:
                    rle = dict(rec['segmentation'])
                elif rec.get('counts') is not None:
                    rle = {'counts': rec['counts']}
                else:
                    return (inst, None, 'fail')

                # 3) size(H,W) 강제
                if 'size' not in rle or rle['size'] is None:
                    rle['size'] = [int(img_h), int(img_w)]
                else:
                    try:
                        rle['size'] = [int(rle['size'][0]), int(rle['size'][1])]
                    except Exception:
                        rle['size'] = [int(img_h), int(img_w)]

                # 4) decode (pycocotools만 사용)
                try:
                    m = maskUtils.decode(rle)
                    if m is None:
                        return (inst, None, 'fail')
                    if m.ndim == 3:
                        m = m[:, :, 0]
                    mu8 = (m.astype(np.uint8) * 255)
                    if mu8.size == 0 or int(mu8.sum()) == 0:
                        # 디코드 됐는데 비어있으면 실패 취급(컨투어 못 잡힘)
                        return (inst, None, 'fail')
                    return (inst, mu8, 'ok')
                except Exception:
                    return (inst, None, 'fail')


            merged_mask = None
            found = 0
            ok = 0
            fail = 0
            miss = 0

            # decode 병렬
            with ThreadPoolExecutor(max_workers=MASK_DECODE_WORKERS) as ex:
                futs = [ex.submit(fetch_and_decode, inst, merged_df, bd_norm, img_h, img_w, masks_bd_index, masks_uid_index) for inst in inst_ids]
                for fu in as_completed(futs):
                    inst, mu8, st = fu.result()
                    found += 1
                    if st == 'miss':
                        miss += 1
                        continue
                    if st == 'fail':
                        fail += 1
                        continue
                    ok += 1
                    if merged_mask is None:
                        merged_mask = (mu8 > 0)
                    else:
                        merged_mask |= (mu8 > 0)

            mask_found_total += found
            mask_decode_success += ok
            mask_decode_fail += fail
            mask_lookup_miss += miss

            if merged_mask is not None and ok > 0:
                mask_used_slots += 1
                shp = calc_shape_and_extra_from_mask((merged_mask.astype(np.uint8) * 255))

                # ✅ 안전 덮어쓰기: 계산값이 finite일 때만 덮어씀
                if is_finite(shp.get('area_px')):
                    area_px = float(shp['area_px'])
                if is_finite(shp.get('perimeter_px')):
                    perimeter_px = float(shp['perimeter_px'])
                if is_finite(shp.get('circularity')):
                    circularity = float(shp['circularity'])
                if is_finite(shp.get('solidity')):
                    solidity = float(shp['solidity'])
                if is_finite(shp.get('concavity')):
                    concavity = float(shp['concavity'])
                if is_finite(shp.get('bbox_w')):
                    bbox_w = float(shp['bbox_w'])
                if is_finite(shp.get('bbox_h')):
                    bbox_h = float(shp['bbox_h'])

                if is_finite(shp.get('curvature')):
                    curvature = float(shp['curvature'])
                if is_finite(shp.get('roughness')):
                    roughness = float(shp['roughness'])
                if is_finite(shp.get('bottom_flatness')):
                    bottom_flatness = float(shp['bottom_flatness'])
                if is_finite(shp.get('core_prominence')):
                    core_prominence = float(shp['core_prominence'])

        # aspect_ratio
        aspect_ratio = float(bbox_w / bbox_h) if (is_finite(bbox_w) and is_finite(bbox_h) and bbox_h > 0) else np.nan

        # cylinder 기본값(나중에 scale_map merge로 덮어씀)
        cyl_ok = False
        cyl_diam_px = 0.0

        # 최종 row (열이 통째로 비는 상황 방지: 기본 0)
        r = {c: 0 for c in REQUIRED_COLS}
        r.update({
            'image_path': bed_image_path,
            'base_key': bed_base_key,
            'lettuce_id': sid,
            'bed_date': bed_date,
            'bed_date_clean': bed_date,
            'date': bed_date_val,
            'position_group': want_row,
            'n_instances': n_instances,
            'conf': conf,
            'brightness_mean': brightness,
            'blur_score': blur,
            'area_px': area_px,
            'area_cm2': area_cm2,
            'px_per_mm_x': px_per_mm_x,
            'px_per_mm_y': px_per_mm_y,
            'mm_per_px': mm_per_px,
            'px_per_mm': px_per_mm,
            'cyl_ok': cyl_ok,
            'cyl_diam_px': cyl_diam_px,
            'front_height_cm': front_height_cm,
            'area_front': area_front,
            'bbox_w': bbox_w,
            'bbox_h': bbox_h,
            'perimeter_px': perimeter_px,
            'circularity': circularity,
            'solidity': solidity,
            'concavity': concavity,
            'curvature': curvature,
            'roughness': roughness,
            'bottom_flatness': bottom_flatness, # Fixed: changed 'bf' to 'bottom_flatness'
            'core_prominence': core_prominence,   # Fixed: changed 'cp' to 'core_prominence'
            'aspect_ratio': aspect_ratio,
            'best_instance': int(seed_id),
        })

        rows_out.append(r)
        curr_centroids_for_next[sid] = (cx, cy)

    missing_slot_count = int(sum([1 for r in rows_out if int(r.get('n_instances', 0) or 0) == 0]))

    used_costs = [mapping_cost.get(s, np.nan) for s in SLOT_IDS if s in mapping_cost]
    used_costs = [c for c in used_costs if is_finite(c)]

    qc = {
        'bed_date': bed_date,
        'missing_slot_count': missing_slot_count,
        'row_mismatch_filtered': int(row_mismatch_filtered),
        'hungarian_used_candidates': int(len(cand)),
        'hungarian_mean_cost': float(np.mean(used_costs)) if used_costs else np.nan,
        'hungarian_max_cost': float(np.max(used_costs)) if used_costs else np.nan,
        'top12_coverage': float(used_seed_area_sum / total_area_all) if total_area_all > 0 else np.nan,
        'mask_found_total': int(mask_found_total),
        'mask_decode_success': int(mask_decode_success),
        'mask_decode_fail': int(mask_decode_fail),
        'mask_lookup_miss': int(mask_lookup_miss),
        'mask_used_slots': int(mask_used_slots),
    }

    res_df = pd.DataFrame(rows_out)
    qc_df = pd.DataFrame([qc])

    # prev용 centroid 저장(엑셀에는 안 넣어도 됨)
    res_df['centroid_x'] = np.nan
    res_df['centroid_y'] = np.nan
    for i, rr in res_df.iterrows():
        sid = rr['lettuce_id']
        xy = curr_centroids_for_next.get(sid)
        if xy is not None:
            res_df.at[i, 'centroid_x'] = xy[0]
            res_df.at[i, 'centroid_y'] = xy[1]

    return res_df, qc_df, curr_centroids_for_next


# =========================
# 8) main
# =========================

def main():
    print(f"[{now_str()}] Load mid parquet: {MID_PARQUET_PATH}")
    df = pd.read_parquet(MID_PARQUET_PATH)

    # 필수 컬럼 체크(네 mid 기준)
    need_cols = ['bed_id','date','bed_date','img_path','warp_W','warp_H','instance_id','instance_uid',
                'conf','centroid_x','centroid_y','bbox_w','bbox_h','brightness_mean','blur_score','area_px',
                'area_front','area_cm2','perimeter_px','solidity','circularity','row_flag','mm_per_px','px_per_mm','front_height_cm']
    for c in ['bed_date','instance_id','centroid_x','centroid_y','area_px','row_flag']:
        if c not in df.columns:
            raise ValueError(f"mid parquet에 필수 컬럼 누락: {c}")

    # img size
    img_w = int(df['warp_W'].iloc[0])
    img_h = int(df['warp_H'].iloc[0])
    print(f"[{now_str()}] Image size from parquet: {img_w}x{img_h}")

    # px_per_mm_x/y 없으면 px_per_mm로 채움
    if 'px_per_mm_x' not in df.columns:
        df['px_per_mm_x'] = df.get('px_per_mm', np.nan)
    if 'px_per_mm_y' not in df.columns:
        df['px_per_mm_y'] = df.get('px_per_mm', np.nan)

    # masks index
    print(f"[{now_str()}] Load masks jsonl: {MASK_JSONL_PATH}")
    masks_bd_index, masks_uid_index = load_masks_index(MASK_JSONL_PATH)
    print(f"[{now_str()}] masks_index (bed_date,instance_id): {len(masks_bd_index):,} | instance_uid: {len(masks_uid_index):,}")

    # bed_date list
    bed_dates = sorted(df['bed_date'].astype(str).unique().tolist())
    print(f"[{now_str()}] bed_date count: {len(bed_dates):,}")

    # group
    groups = {bd: g.copy() for bd, g in df.groupby(df['bed_date'].astype(str), sort=False)}

    slot_centers = build_slot_centers(img_w, img_h)

    part_dir = os.path.join(OUT_DIR, "parts")
    os.makedirs(part_dir, exist_ok=True)

    done_set = set()
    if RESUME:
        for p in glob.glob(os.path.join(part_dir, "result_part_*.parquet")):
            try:
                tmp = pd.read_parquet(p, columns=['bed_date'])
                done_set.update(tmp['bed_date'].astype(str).unique().tolist())
            except Exception:
                pass
        print(f"[{now_str()}] RESUME on: already done bed_dates = {len(done_set):,}")

    t0 = time.time()
    results_parts, qc_parts = [], []
    save_idx = 0
    processed = 0

    prev_summary = None

    def load_prev_from_parts(bd: str) -> Optional[pd.DataFrame]:
        for p in sorted(glob.glob(os.path.join(part_dir, "result_part_*.parquet"))):
            try:
                tmp = pd.read_parquet(p)
                sub = tmp[tmp['bed_date'].astype(str) == bd]
                if len(sub):
                    return sub
            except Exception:
                continue
        return None

    for i, bd in enumerate(bed_dates):
        elapsed = time.time() - t0
        rate = elapsed / max(1, i)
        eta = rate * (len(bed_dates) - i)
        if i % 20 == 0:
            print(f"[{now_str()}] {i}/{len(bed_dates)} | elapsed {fmt_sec(elapsed)} | ETA {fmt_sec(eta)}")

        if RESUME and bd in done_set:
            prev_summary = load_prev_from_parts(bd)
            continue

        df_bd = groups.get(bd)
        if df_bd is None or len(df_bd) == 0:
            continue

        res_df, qc_df, curr_centroids = process_one_beddate(
            bed_date=bd,
            df_bd=df_bd,
            slot_centers=slot_centers,
            img_w=img_w,
            img_h=img_h,
            masks_bd_index=masks_bd_index,
            masks_uid_index=masks_uid_index,
            prev_summary=prev_summary,
        )

        prev_summary = res_df.copy()

        results_parts.append(res_df)
        qc_parts.append(qc_df)
        processed += 1

        if processed % CHUNK_BEDDATES == 0:
            part_res = pd.concat(results_parts, ignore_index=True)
            part_qc = pd.concat(qc_parts, ignore_index=True)

            p_res = os.path.join(part_dir, f"result_part_{save_idx:05d}.parquet")
            p_qc  = os.path.join(part_dir, f"qc_part_{save_idx:05d}.parquet")

            part_res.to_parquet(p_res, index=False)
            part_qc.to_parquet(p_qc, index=False)
            print(f"[{now_str()}] saved parts: {p_res} | {p_qc}")

            results_parts, qc_parts = [], []
            save_idx += 1

    if results_parts:
        part_res = pd.concat(results_parts, ignore_index=True)
        part_qc = pd.concat(qc_parts, ignore_index=True)
        p_res = os.path.join(part_dir, f"result_part_{save_idx:05d}.parquet")
        p_qc  = os.path.join(part_dir, f"qc_part_{save_idx:05d}.parquet")
        part_res.to_parquet(p_res, index=False)
        part_qc.to_parquet(p_qc, index=False)
        print(f"[{now_str()}] saved final parts: {p_res} | {p_qc}")

    # concat
    res_all, qc_all = [], []
    for p in sorted(glob.glob(os.path.join(part_dir, "result_part_*.parquet"))):
        res_all.append(pd.read_parquet(p))
    for p in sorted(glob.glob(os.path.join(part_dir, "qc_part_*.parquet"))):
        qc_all.append(pd.read_parquet(p))

    res_all = pd.concat(res_all, ignore_index=True)
    qc_all = pd.concat(qc_all, ignore_index=True) if qc_all else pd.DataFrame(columns=QC_COLS)

    # scale_map merge(선택): bed_id 기준
    if isinstance(SCALE_MAP_CSV, str) and SCALE_MAP_CSV.strip() and os.path.exists(SCALE_MAP_CSV):
        try:
            sm = pd.read_csv(SCALE_MAP_CSV)
            # 기대: bed_id, px_per_mm_x/px_per_mm_y/cyl_ok/cyl_diam_px 등
            # bed_id 컬럼명 보정
            if 'bed_id' not in sm.columns:
                for cand in ['base_key','bed','bedid']:
                    if cand in sm.columns:
                        sm = sm.rename(columns={cand: 'bed_id'})
                        break
            sm['bed_id_norm'] = sm['bed_id'].astype(str).apply(norm_key)
            res_all['bed_id_norm'] = res_all['base_key'].astype(str).apply(norm_key)
            res_all = res_all.merge(
                sm.drop_duplicates('bed_id_norm'),
                how='left',
                left_on='bed_id_norm',
                right_on='bed_id_norm',
                suffixes=('', '_sm')
            )
            # scale_map 값으로 덮어쓰기(있을 때만)
            for col in ['cyl_ok','cyl_diam_px','px_per_mm_x','px_per_mm_y']:
                sm_col = f"{col}_sm"
                if sm_col in res_all.columns:
                    mask = res_all[sm_col].notna()
                    res_all.loc[mask, col] = res_all.loc[mask, sm_col]
            # 정리
            drop_cols = [c for c in res_all.columns if c.endswith('_sm')] + ['bed_id_norm']
            res_all = res_all.drop(columns=[c for c in drop_cols if c in res_all.columns])
        except Exception as e:
            print(f"[WARN] scale_map merge failed: {e}")

    # 정렬
    res_all['lettuce_id'] = pd.Categorical(res_all['lettuce_id'], categories=SLOT_IDS, ordered=True)
    res_all = res_all.sort_values(['bed_date', 'lettuce_id']).reset_index(drop=True)

    # 엑셀
    with pd.ExcelWriter(FINAL_XLSX, engine='openpyxl') as writer:
        res_all[REQUIRED_COLS].to_excel(writer, sheet_name='metrics', index=False)
        qc_all[QC_COLS].to_excel(writer, sheet_name='qc', index=False)

        cfg = {
            'MID_PARQUET_PATH': MID_PARQUET_PATH,
            'MASK_JSONL_PATH': MASK_JSONL_PATH,
            'SCALE_MAP_CSV': SCALE_MAP_CSV,
            'OUT_DIR': OUT_DIR,
            'CHUNK_BEDDATES': CHUNK_BEDDATES,
            'RESUME': RESUME,
            'TOPK_SEEDS': TOPK_SEEDS,
            'ALPHA_SLOT': ALPHA_SLOT,
            'BETA_PREV': BETA_PREV,
            'USE_SLOT_GATE': USE_SLOT_GATE,
            'MERGE_DIST_RATIO': MERGE_DIST_RATIO,
            'OUTLIER_LOW': OUTLIER_LOW,
            'OUTLIER_HIGH': OUTLIER_HIGH,
            'AREA_MIN_FOR_OUTLIER': AREA_MIN_FOR_OUTLIER,
            'RECALC_SHAPE_FROM_MASK': RECALC_SHAPE_FROM_MASK,
            'HAS_SCIPY': _HAS_SCIPY,
            'HAS_CV2': _HAS_CV2,
            'HAS_PYCOCO': _HAS_PYCOCO,
            'MASK_DECODE_WORKERS': MASK_DECODE_WORKERS,
        }
        pd.DataFrame([cfg]).to_excel(writer, sheet_name='run_config', index=False)

    print(f"[{now_str()}] DONE -> {FINAL_XLSX}")

if __name__ == "__main__":
    main()


SyntaxError: closing parenthesis ')' does not match opening parenthesis '[' on line 448 (ipython-input-507012177.py, line 451)